In [ ]:
pip install pandas requests tqdm

In [ ]:
import pandas as pd
import requests
import time
import os
import re
from tqdm import tqdm

In [ ]:
import pandas as pd
import os

# ==========================================
# ⚙️ 설정
# ==========================================
input_csv_path = "./bq-results-20251113-102522-1763029560415.csv" 
output_csv_path = "./unique_cited_patents_list_bq.csv" # BigQuery용 파일명 변경

def main():
    if not os.path.exists(input_csv_path):
        print("파일이 없습니다.")
        return

    # 파일 읽기
    try:
        df = pd.read_csv(input_csv_path, encoding='utf-8-sig') # 읽을 땐 sig 허용
    except:
        df = pd.read_csv(input_csv_path, encoding='cp949')

    if 'cited_patent_ids' not in df.columns:
        print("'cited_patent_ids' 컬럼이 없습니다.")
        return

    # ID 추출 및 정제
    print("ID 추출 중...")
    citation_set = set()
    
    for citations in df['cited_patent_ids'].dropna():
        # 세미콜론 분리
        ids = [c.strip() for c in str(citations).split(';') if c.strip()]
        for pid in ids:
            # 혹시 모를 빈 문자열이나 이상한 기호 제거
            clean_id = pid.replace('"', '').replace("'", "").strip()
            if clean_id:
                citation_set.add(clean_id)

    unique_list = sorted(list(citation_set))
    
    print(f"✅ 총 {len(unique_list)}개의 유니크 ID 추출 완료")

    # 저장 (BigQuery 친화적 설정)
    result_df = pd.DataFrame(unique_list, columns=['id'])
    
    # 🚨 중요: encoding='utf-8' (sig 뺌)
    result_df.to_csv(output_csv_path, index=False, encoding='utf-8')
    
    print(f"💾 BigQuery용 파일 저장 완료: {output_csv_path}")
    print("이 파일을 다시 업로드해보세요 (헤더 건너뛰기: 1 설정 필수)")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import glob

# 1. 파일 목록 정의 (현재 폴더에 있는 'gp-search*.csv' 패턴의 모든 파일)
# 실제 사용 시에는 파일들이 있는 경로를 지정해주시면 됩니다.
file_pattern = "gp-search-*.xlsx"
file_names = glob.glob(file_pattern)

dfs = []

# 2. 각 파일을 읽어서 리스트에 저장
for filename in file_names:
    try:
        # 첫 번째 줄(검색 URL)을 건너뛰고(skiprows=1) 데이터를 읽어옵니다.
        df = pd.read_excel(filename, skiprows=1)
        dfs.append(df)
    except Exception as e:
        print(f"Error reading {filename}: {e}")

# 3. 데이터프레임 합치기 및 저장
if dfs:
    merged_df = pd.concat(dfs, ignore_index=True)
    merged_filename = "merged_patents.csv"
    merged_df.to_csv(merged_filename, index=False, encoding='utf-8-sig')
    
    print(f"성공적으로 {len(dfs)}개의 파일을 합쳤습니다.")
    print(f"총 데이터 개수: {len(merged_df)}")
    print(f"저장된 파일명: {merged_filename}")
else:
    print("합칠 파일이 없습니다.")

In [ ]:
import pandas as pd
import networkx as nx
from collections import defaultdict
import itertools
import os
from tqdm import tqdm

# 1. 데이터 형식을 처리하기 위한 헬퍼 함수 (수정됨)
def process_and_extract_ids(ref_string: str) -> list:
    """
    'US-319296-A; US-668879-A; ... ; ;' 형태의 문자열을 파싱하여 리스트로 반환합니다.
    세미콜론(;)으로 분리하고 공백을 제거하며, 빈 항목은 제외합니다.
    """
    if not isinstance(ref_string, str) or not ref_string.strip():
        return []
    
    # 세미콜론으로 분리 후 앞뒤 공백 제거, 빈 문자열 필터링
    id_list = [ref.strip() for ref in ref_string.split(';') if ref.strip()]
    return id_list

# 2. 네트워크 구축 함수 (수정됨)
def build_co_citation_network(papers_df: pd.DataFrame, min_co_citations: int = 2) -> nx.Graph:
    # 데이터 전처리: 문자열을 리스트로 변환
    # tqdm을 사용하여 처리 진행률 표시 (데이터가 많을 경우를 대비)
    tqdm.pandas(desc="Processing cited_patent_ids")
    papers_df['cited_patent_ids'] = papers_df['cited_patent_ids'].apply(process_and_extract_ids)
    
    # {original_id: [cited_id1, cited_id2, ...]} 딕셔너리 생성
    # 사용자가 요청한대로 노드 식별을 위해 original_id를 Key로 사용
    source_to_references = {
        row['original_id']: row['cited_patent_ids'] 
        for index, row in papers_df.iterrows()
    }
    
    # --- 기존 로직 유지 (공동 인용 계산) ---
    
    # 1단계: 각 인용 특허(Reference)가 어떤 원본 특허(Source)들에 의해 인용되었는지 매핑
    reference_cited_by = defaultdict(set)
    for source_id, refs in source_to_references.items():
        for ref_id in refs:
            reference_cited_by[ref_id].add(source_id)
            
    # 각 인용 특허의 총 피인용 횟수 계산 (Jaccard 계수 분모용)
    reference_citation_counts = {ref_id: len(sources) for ref_id, sources in reference_cited_by.items()}
    
    # 2단계: 공동 인용 횟수 계산 (Co-citation Counts)
    co_citation_counts = defaultdict(int)
    for source_id, references in tqdm(source_to_references.items(), desc="Counting co-citations"):
        if len(references) >= 2:
            # 한 특허 내에서 인용된 특허들의 모든 조합(쌍) 생성
            for ref1, ref2 in itertools.combinations(sorted(references), 2):
                edge_key = tuple(sorted((ref1, ref2)))
                co_citation_counts[edge_key] += 1
                
    # 3단계: 그래프 생성 및 가중치(Association Strength) 계산
    G = nx.Graph()
    for (ref_id1, ref_id2), count in tqdm(co_citation_counts.items(), desc="Building Graph"):
        if count >= min_co_citations:
            citations_ref1 = reference_citation_counts.get(ref_id1, 0)
            citations_ref2 = reference_citation_counts.get(ref_id2, 0)
            
            # Association Strength = Co-citation Count / (Total Citations A + Total Citations B - Co-citation Count)
            denominator = citations_ref1 + citations_ref2 - count
            if denominator > 0:
                association_strength = count / denominator
                G.add_edge(ref_id1, ref_id2, weight=association_strength)
                
    return G

# 3. 가장 큰 연결 요소 추출 함수 (수정 없음)
def get_largest_connected_component(graph: nx.Graph) -> nx.Graph:
    if not graph.nodes:
        return nx.Graph()
    connected_components = list(nx.connected_components(graph))
    if not connected_components:
        return nx.Graph()
    largest_component_nodes = max(connected_components, key=len)
    return graph.subgraph(largest_component_nodes).copy()

# --- 메인 실행 블록 ---
if __name__ == "__main__":
    # 1. 파일 로드 (경로 및 로직 수정)
    file_path = "./bq-results-20251113-102522-1763029560415.csv"
    
    if os.path.exists(file_path):
        print(f"Loading {file_path}...")
        # referenced_works 대신 cited_patent_ids 컬럼이 있으므로 해당 컬럼을 문자열로 읽음
        papers_df = pd.read_csv(file_path, dtype={'cited_patent_ids': str, 'original_id': str}, low_memory=False)
        print(f"총 {len(papers_df)} 개의 특허 데이터를 로드했습니다.")
    else:
        print(f"Error: 파일을 찾을 수 없습니다. 경로를 확인해주세요: {file_path}")
        exit()
    
    # 2. 공동 인용 네트워크 구축
    print("\n--- 공동 인용 네트워크 구축 시작 ---")
    # min_co_citations=1로 설정하여 최대한 많은 연결을 포함 (필요시 조절 가능)
    co_citation_graph = build_co_citation_network(papers_df, min_co_citations=1)
    
    print(f"\n--- 전체 공동 인용 네트워크 정보 (LCC 추출 전) ---")
    print(f"  노드 수: {co_citation_graph.number_of_nodes()}")
    print(f"  엣지 수: {co_citation_graph.number_of_edges()}")
    
    # 3. 가장 큰 연결 요소(LCC) 추출
    lcc_graph = get_largest_connected_component(co_citation_graph)
    print(f"\n가장 큰 연결 요소(LCC) 네트워크 정보:")
    print(f"  노드 수: {lcc_graph.number_of_nodes()}")
    print(f"  엣지 수: {lcc_graph.number_of_edges()}")

    # 4. 네트워크 필터링 (목표 노드 수에 맞춰서 Weight 기반 필터링)
    target_node_count = 5000  # 목표 노드 수 (필요시 수정)
    final_graph = lcc_graph

    if lcc_graph.number_of_nodes() > 0:
        print(f"\n--- 네트워크 필터링 시작 (목표 노드 수: {target_node_count} 근사) ---")
        
        weights = [data['weight'] for u, v, data in lcc_graph.edges(data=True)]
        # 중복 제거 및 내림차순 정렬
        sorted_unique_weights = sorted(list(set(weights)), reverse=True)
        
        best_graph = lcc_graph
        min_difference = abs(lcc_graph.number_of_nodes() - target_node_count)
        best_threshold = 0.0
        
        # Threshold를 높여가며 최적의 그래프 탐색
        for threshold in tqdm(sorted_unique_weights, desc="Finding Best Threshold"):
            # 현재 Threshold 이상인 엣지만 유지
            edges_to_keep = [(u, v) for u, v, data in lcc_graph.edges(data=True) if data['weight'] >= threshold]
            
            # 엣지가 없으면 건너뜀
            if not edges_to_keep: continue

            # 서브 그래프 생성 및 LCC 추출
            sub_graph = lcc_graph.edge_subgraph(edges_to_keep)
            sub_lcc = get_largest_connected_component(sub_graph)
            current_node_count = sub_lcc.number_of_nodes()
            
            if current_node_count == 0: continue

            current_difference = abs(current_node_count - target_node_count)
            
            # 목표 노드 수와 차이가 더 작으면 업데이트
            if current_difference < min_difference:
                min_difference = current_difference
                best_graph = sub_lcc
                best_threshold = threshold
            # 차이가 같으면 노드 수가 더 많은 쪽을 선택 (데이터 손실 최소화)
            elif current_difference == min_difference and current_node_count > best_graph.number_of_nodes():
                 best_graph = sub_lcc
                 best_threshold = threshold
            
            # (옵션) 만약 노드 수가 목표보다 훨씬 작아지면 조기 종료할 수도 있음
            if current_node_count < target_node_count * 0.5:
                break

        final_graph = best_graph
        print(f"\n최적 Threshold 탐색 완료! 적용된 Threshold: {best_threshold:.4f}")
        print(f"  최종 노드 수: {final_graph.number_of_nodes()} (목표와의 차이: {min_difference})")
        print(f"  최종 엣지 수: {final_graph.number_of_edges()}")

    # 5. 파일 저장
    if final_graph.number_of_nodes() > 0:
        output_gexf_file = "patent_co_citation_network_filtered.gexf" 
        nx.write_gexf(final_graph, output_gexf_file)
        print(f"\n필터링된 공동 인용 네트워크가 '{output_gexf_file}' 파일로 저장되었습니다.")
    else:
        print("\n생성된 그래프의 노드가 없어 파일을 저장하지 않았습니다.")